# 03 — Real two-GPU throughput: converging queue vs the alternatives

**Run on the CUDA host** (`uv sync --extra dev --extra gpu`, then
`uv run jupyter lab`).
This notebook decides whether the project was worth building; keep its
executed outputs committed — they are the evidence. (Committed unexecuted
from the CPU dev machine; execute here and re-commit.)

Four configurations, same inputs, same model:

1. fast GPU alone (TEI-equivalent baseline)
2. both GPUs, static 50/50 item split
3. both GPUs, static split weighted by configured device weights
4. both GPUs, converging queue (embedx as built)

Measurement hygiene, all required: correctness asserted before any timing,
warmup runs discarded, `torch.cuda.synchronize()` on every device around
every timed region, medians over several runs with spread, and an explicit
record of anything else holding VRAM during the run.

In [ ]:
import subprocess
import threading
import time

import numpy as np
import torch

print(torch.cuda.get_arch_list())

# --- Environment record: contaminated numbers that look clean are worse ---
# --- than no numbers. If Ollama (or anything) holds VRAM, it shows here. ---
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} {torch.cuda.get_device_name(i)}  "
          f"free={free / 2**30:.1f} GiB / total={total / 2**30:.1f} GiB  "
          f"(anything missing from 'free' is held by another process)")
OTHER_VRAM_USERS = "NONE"  # <- EDIT: state what nvidia-smi shows, honestly.
print(f"declared other VRAM users during this run: {OTHER_VRAM_USERS}")

In [ ]:
from embedx.backend.hf import HFBackend, TokenLengthCache
from embedx.config import Settings
from embedx.engine import Engine, make_batches
from embedx.gpu.budgets import device_budgets
from embedx.gpu.discovery import discover_devices, rank_devices

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
settings = Settings(model_id=MODEL, pooling="mean", max_batch_tokens=16384, max_seq_len=512)

infos = discover_devices(settings.devices)
ranked = rank_devices(infos, settings.device_weights)
budgets = device_budgets(ranked, settings.max_batch_tokens, settings.device_batch_tokens)
assert len(ranked) >= 2, "this notebook needs two GPUs"
for d in ranked:
    print(f"[{d.index}] {d.name}  weight={d.weight:.3f}  budget={budgets[d.index]}")

cache = TokenLengthCache()
backends = [
    HFBackend(MODEL, device_index=d.index, pooling=settings.pooling,
              normalize=settings.normalize, dtype=settings.dtype,
              max_seq_length=settings.max_seq_len, length_cache=cache)
    for d in ranked
]
length_fn = backends[0].length_fn
engine = Engine(backends, ranked, settings, length_fn=length_fn)

# Corpus: real texts, real token lengths. The bare id "ag_news" stopped
# resolving in datasets>=5 ("Repository id must be 'namespace/name'"), which
# silently demoted every number in this notebook to repeated lorem ipsum.
# Pinned now, and the fallback has to be asked for explicitly.
AG_NEWS = "fancyzhx/ag_news"
ALLOW_SYNTHETIC = False  # True only for a corpus-free plumbing check

try:
    from datasets import load_dataset

    texts = [r["text"] for r in load_dataset(AG_NEWS, split="train[:8000]")]
    CORPUS_LABEL = f"{AG_NEWS} train[:8000]"
except Exception as exc:  # noqa: BLE001
    if not ALLOW_SYNTHETIC:
        raise RuntimeError(
            f"real corpus unavailable ({type(exc).__name__}: {exc}). Throughput "
            "numbers measured on synthetic text are not publishable; set "
            "ALLOW_SYNTHETIC=True above to check the plumbing anyway."
        ) from exc
    rng = np.random.default_rng(7)
    lens = np.clip(rng.lognormal(4.0, 0.9, size=8000).astype(int), 10, 3000)
    texts = ["lorem ipsum " * max(1, n // 12) for n in lens]
    CORPUS_LABEL = f"!! SYNTHETIC — NOT REAL TEXT, DO NOT PUBLISH ({type(exc).__name__})"

print(f"corpus: {CORPUS_LABEL}")
token_lengths = [length_fn(t) for t in texts]
TOTAL_TOKENS = sum(token_lengths)
print(f"n={len(texts)}  total tokens={TOTAL_TOKENS}")

In [ ]:
def sync_all():
    for d in ranked:
        torch.cuda.synchronize(d.index)


class Recording:
    # Wraps a backend to record per-device items/tokens/last-finish.
    def __init__(self, inner, device_index):
        self.inner, self.device_index = inner, device_index
        self.dim = inner.dim
        self.reset()

    def reset(self):
        self.items = self.tokens = 0
        self.last_end = 0.0

    def embed(self, batch_texts):
        out = self.inner.embed(batch_texts)
        torch.cuda.synchronize(self.device_index)
        self.items += len(batch_texts)
        self.tokens += sum(length_fn(t) for t in batch_texts)
        self.last_end = time.perf_counter()
        return out


recorders = [Recording(b, d.index) for b, d in zip(backends, ranked)]
rec_engine = Engine(recorders, ranked, settings, length_fn=length_fn)


def static_bounds(shares):
    """Contiguous split of the length-sorted list, balanced on TOKENS.

    Splitting on item count instead makes these baselines a strawman. The
    list is sorted by length, so a contiguous 50/50 split by items handed
    the slow A400 81% of the tokens, and the "weighted" split still gave it
    30% of the tokens for 8% of the items. The static configs would then
    lose to the converging queue for a reason that has nothing to do with
    scheduling. Boundaries are taken off the cumulative token curve, so the
    intended share is the delivered share.
    """
    shares = np.asarray(shares, dtype=float)
    shares = shares / shares.sum()
    targets = np.cumsum(shares) * TOTAL_TOKENS
    order = sorted(range(len(texts)), key=lambda i: token_lengths[i])
    bounds, running, dev = [0], 0, 0
    for pos, i in enumerate(order):
        running += token_lengths[i]
        while dev < len(shares) - 1 and running >= targets[dev]:
            bounds.append(pos + 1)
            dev += 1
    while len(bounds) <= len(shares):
        bounds.append(len(order))
    bounds[-1] = len(order)
    return order, bounds


def describe_split(shares):
    order, bounds = static_bounds(shares)
    rows = []
    for w in range(len(shares)):
        chunk = order[bounds[w]:bounds[w + 1]]
        tok = sum(token_lengths[i] for i in chunk)
        rows.append((len(chunk), tok, tok / TOTAL_TOKENS))
    return rows


def run_static_split(shares):
    order, bounds = static_bounds(shares)
    threads = []
    for w, (rec, dev) in enumerate(zip(recorders, ranked)):
        chunk = [(i, texts[i]) for i in order[bounds[w]:bounds[w + 1]]]

        def work(rec=rec, dev=dev, chunk=chunk):
            for batch in make_batches(chunk, budgets[dev.index], length_fn=length_fn):
                rec.embed([t for _, t in batch])

        threads.append(threading.Thread(target=work))
    for t in threads:
        t.start()
    for t in threads:
        t.join()


def timed(fn, runs=5, warmup=2):
    # Warmup pays CUDA context + autotune; medians over `runs` with spread.
    for _ in range(warmup):
        fn()
    times, idles = [], []
    for _ in range(runs):
        for r in recorders:
            r.reset()
        sync_all()
        t0 = time.perf_counter()
        fn()
        sync_all()
        end = time.perf_counter()
        times.append(end - t0)
        # Idle = how long a device sat after its last batch while the others
        # were still working. None, not NaN, when a device did no work at
        # all: "never asked to work" is a different claim from "never idle".
        idles.append([end - r.last_end if r.items else None for r in recorders])
    med = float(np.median(times))

    def med_idle(k):
        vals = [row[k] for row in idles if row[k] is not None]
        return float(np.median(vals)) if vals else None

    # items/tokens describe the LAST run; makespan is the median of all runs.
    return {"median_s": med,
            "iqr": (float(np.percentile(times, 25)), float(np.percentile(times, 75))),
            "tokens_per_s": TOTAL_TOKENS / med,
            "runs_s": times,
            "per_device": [{"index": d.index, "name": d.name, "items": r.items,
                            "tokens": r.tokens, "idle_s": med_idle(k)}
                           for k, (r, d) in enumerate(zip(recorders, ranked))]}

In [ ]:
# --- Correctness FIRST. A fast wrong answer is worth nothing. -------------
# The production dtype is lossy: dtype=auto resolves to bfloat16 on both
# cards (capability 12.0 and 8.6, both >= 8). bfloat16 keeps 8 mantissa
# bits, so near a component magnitude of 0.1 one ULP is ~2.4e-4 and two
# different architectures legitimately disagree by a few ULPs. The previous
# assert_allclose(atol=1e-3) could not hold no matter how correct the engine
# was — it was measuring bfloat16, not embedx.
#
# Assert what actually has to be true instead, which is strictly stronger:
#   (a) in float32 the sharded engine must be BIT-IDENTICAL to a single GPU,
#       leaving no room for a pooling, masking, sharding or ordering error;
#   (b) in the production dtype, order is preserved (every row's nearest
#       neighbour is its own reference) and agreement is within a few ULPs.
probe = texts[:256]

# (a) exactness in float32 -------------------------------------------------
fp32 = Settings(model_id=MODEL, pooling="mean", max_batch_tokens=16384,
                max_seq_len=512, dtype="float32")
fp32_cache = TokenLengthCache()
fp32_backends = [HFBackend(MODEL, device_index=d.index, pooling=fp32.pooling,
                           normalize=fp32.normalize, dtype=fp32.dtype,
                           max_seq_length=fp32.max_seq_len, length_cache=fp32_cache)
                 for d in ranked]
fp32_engine = Engine(fp32_backends, ranked, fp32, length_fn=fp32_backends[0].length_fn)
exact_ref = fp32_backends[0].embed(probe)
exact_eng = fp32_engine.embed(probe)
assert np.array_equal(exact_eng, exact_ref), "float32: sharded engine != single GPU"
print(f"PASS float32: sharded engine is BIT-IDENTICAL to one GPU "
      f"({exact_eng.shape[0]} rows x {exact_eng.shape[1]} dims)")
del fp32_backends, fp32_engine, exact_ref, exact_eng
torch.cuda.empty_cache()


# (b) ordering + agreement in the production dtype -------------------------
def unit(m):
    return m / np.linalg.norm(m, axis=1, keepdims=True)


reference = backends[0].embed(probe)
converged = engine.embed(probe)
sim = unit(converged) @ unit(reference).T
self_sim = np.diag(sim)
misordered = int((sim.argmax(axis=1) != np.arange(len(probe))).sum())
assert misordered == 0, f"ORDERING VIOLATED: {misordered} rows match another input"
assert self_sim.min() > 0.9999, f"cosine floor breached: {self_sim.min():.6f}"
print(f"PASS {backends[0]._dtype}: order preserved for all {len(probe)} rows; "
      f"min self-cosine {self_sim.min():.6f}; max |diff| "
      f"{np.abs(converged - reference).max():.6f} (cross-architecture bfloat16 noise)")

In [ ]:
weights = np.array([d.weight for d in ranked])
even = [1 / len(ranked)] * len(ranked)
weighted = list(weights / weights.sum())

# Show that each static baseline delivers the token share it claims to.
# Without this the comparison below cannot be trusted.
for label, shares in (("2 static 50/50", even), ("3 static weighted", weighted)):
    detail = "  ".join(f"dev{d.index}: {n} items / {t} tok ({f:.1%})"
                       for (n, t, f), d in zip(describe_split(shares), ranked))
    intended = ", ".join(f"{s:.1%}" for s in shares)
    print(f"{label:22s} intended token share [{intended}] -> {detail}")
print()

configs = {
    "1 fast GPU alone": lambda: [recorders[0].embed([t for _, t in b])
                                 for b in make_batches(list(enumerate(texts)),
                                                       budgets[ranked[0].index],
                                                       length_fn=length_fn)],
    "2 static 50/50": lambda: run_static_split(even),
    "3 static weighted": lambda: run_static_split(weighted),
    "4 converging queue": lambda: rec_engine.embed(texts),
}

results = {}
for name, fn in configs.items():
    results[name] = timed(fn)
    r = results[name]
    print(f"{name:22s} makespan={r['median_s']:.3f}s "
          f"(IQR {r['iqr'][0]:.3f}-{r['iqr'][1]:.3f})  "
          f"{r['tokens_per_s']:,.0f} tok/s")
    for pd in r["per_device"]:
        idle = "n/a (no work)" if pd["idle_s"] is None else f"{pd['idle_s']:.3f}s"
        print(f"    device {pd['index']}: items={pd['items']:5d} "
              f"tokens={pd['tokens']:8d} idle={idle}")

In [ ]:
# Idle time is the direct measure of balance: configs 2 and 3 should show a
# clear per-device gap, config 4 should not.
import matplotlib.pyplot as plt

names = list(results)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(names, [results[n]["median_s"] for n in names])
axes[0].set_ylabel("makespan (s), median of 5")
axes[0].tick_params(axis="x", rotation=20)
width = 0.35
for k, d in enumerate(ranked):
    axes[1].bar(np.arange(len(names)) + k * width,
                [results[n]["per_device"][k]["idle_s"] for n in names],
                width, label=f"device {d.index}")
axes[1].set_xticks(np.arange(len(names)) + width / 2, names, rotation=20)
axes[1].set_ylabel("idle time (s)")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Host-to-device transfer per device. --------------------------------
# The A400 sits on PCIe 3.0 x4; torch's static properties cannot see that.
# This measurement is the empirical justification for device_weights.
SIZE_MB = 256
h2d = {}
host = torch.empty(SIZE_MB * 2**20, dtype=torch.uint8, pin_memory=True)
for d in ranked:
    torch.cuda.synchronize(d.index)
    times = []
    for _ in range(2):  # warmup
        host.to(f"cuda:{d.index}", non_blocking=True)
        torch.cuda.synchronize(d.index)
    for _ in range(7):
        torch.cuda.synchronize(d.index)
        t0 = time.perf_counter()
        host.to(f"cuda:{d.index}", non_blocking=True)
        torch.cuda.synchronize(d.index)
        times.append(time.perf_counter() - t0)
    gbps = (SIZE_MB / 1024) / np.median(times)
    h2d[d.index] = gbps
    print(f"device {d.index} ({d.name}): H2D {gbps:.1f} GiB/s "
          f"(median of 7, {SIZE_MB} MiB pinned)")

## Results — fill in from the run above and keep the outputs committed

State the numbers plainly, including if they are small:

- Speedup of **(4) converging queue** over **(1) fast GPU alone**: `__x`
- Speedup of **(4)** over **(3) weighted static split**: `__x`
- Idle-time gap: config 2 `__s` / config 3 `__s` vs config 4 `__s`
- Measured H2D bandwidth ratio between devices: `__`

If the converging queue does not beat the weighted static split by a
meaningful margin on this hardware, that is the result and it goes here
unedited. The honest negative is more useful than a flattering blank.

## Machine-readable results

The cell below writes everything above to `dev/output/results.json`, so the
README is filled in by transcription rather than by retyping numbers out of
cell output. Re-run it whenever the timings are re-measured.

In [ ]:
# --- Evidence file. README numbers come from here, never from retyping. ---
import json
import pathlib

here = pathlib.Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
out_dir = root / "dev" / "output"
out_dir.mkdir(parents=True, exist_ok=True)

driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version",
                         "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip().splitlines()

payload = {
    "corpus": CORPUS_LABEL,
    "n_texts": len(texts),
    "total_tokens": TOTAL_TOKENS,
    "model": MODEL,
    "pooling": settings.pooling.value,
    "dtype_configured": settings.dtype.value,
    "dtype_in_use": sorted({str(b._dtype) for b in backends}),
    "max_seq_len": settings.max_seq_len,
    "max_batch_tokens": settings.max_batch_tokens,
    "truncated_inputs": sum(b.truncated_count for b in backends),
    "device_budgets": {str(d.index): budgets[d.index] for d in ranked},
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "torch_arch_list": torch.cuda.get_arch_list(),
    "driver": driver[0] if driver else None,
    "devices": [{"index": d.index, "name": d.name, "weight": d.weight,
                 "capability": list(torch.cuda.get_device_capability(d.index))}
                for d in ranked],
    "h2d_gib_s": {str(k): v for k, v in h2d.items()},
    "configs": results,
}
path = out_dir / "results.json"
path.write_text(json.dumps(payload, indent=2, default=str))
print(f"wrote {path}  ({path.stat().st_size} bytes)")
if CORPUS_LABEL.startswith("!!"):
    print("WARNING: synthetic corpus — these numbers must NOT reach the README.")